<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">3. Centralized Data Processing with External Analytics</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 3.2 Demo Powering Downstream Analytics in Snowflake using UC Managed Tables

This demo walks through configuring Snowflake to read managed UC Iceberg tables - specifically the <code>store_sales_iceberg</code> and <code>store_sales_delta_uniform</code> tables built in Module 2 - via Snowflake's <b>Catalog Integration</b> feature against the Unity Catalog Iceberg REST endpoint.

## Learning Objectives

By the end of this demonstration, you will be able to:
- Enable external data access on a Unity Catalog metastore
- Create a Databricks service principal and OAuth secret for an external engine to authenticate as
- Grant the privileges (`EXTERNAL USE SCHEMA`, `USE CATALOG`, `USE SCHEMA`, `SELECT`) required for an external Iceberg REST client
- Configure a Snowflake catalog integration pointing at the UC Iceberg REST endpoint
- Create Snowflake-side Iceberg tables that read directly from UC managed storage with vended credentials
- Demonstrate cross-platform analytics by joining UC `store_sales` data with Snowflake-resident TPC-DS dimensions

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #4a148c; font-size: 1.1em;">Video Demonstration</strong>
            <p style="margin: 8px 0 0 0; color: #333;">In the standard classroom environment, this demo is delivered as a <strong>video walkthrough</strong> because it requires a Snowflake account, a Databricks service principal with OAuth credentials, and customer-managed storage, none of which are available in the lab environment. The notebook below contains the complete working demo for environments that meet the prerequisites listed below. If you have the required infrastructure, you can run it end-to-end.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Required Permissions</strong>
            <p style="margin: 8px 0 0 0; color: #333;">This demo configures account-level objects on both platforms. The person running it needs:</p>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><b>Databricks:</b> <b>Metastore Admin</b> (to enable external data access on the metastore) <i>and</i> <b>Account Admin</b> (to create a service principal and generate an OAuth secret). On a single-workspace account these are typically the same person.</li>
                <li><b>Snowflake:</b> <b><code>ACCOUNTADMIN</code></b> (to create a catalog integration).</li>
            </ul>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #c62828; font-size: 1.1em;">Requires Customer-Managed Storage</strong>
            <p style="margin: 8px 0 0 0; color: #333;">This demo cannot run end-to-end on Databricks-managed serverless storage (e.g. <code>s3://dbstorage-prod-*</code>). External Iceberg engines like Snowflake cannot read those buckets even with credential vending, because Databricks's managed storage is locked down to its own AWS principals at the bucket-policy level.</p>
            <p style="margin: 8px 0 0 0; color: #333;">The instructor demo setup (<code>Classroom-Setup-3-demo</code>) creates the <code>instructor_interop_demo</code> catalog with a <b>customer-managed external location</b> (an S3 path under <i>your own</i> AWS account, registered to UC via a storage credential).</p>
        </div>
    </div>
</div>

## What We're Building

Two managed UC tables on the Databricks side - one Delta + UniForm and one native managed Iceberg - are exposed through the Unity Catalog Iceberg REST endpoint and consumed live from Snowflake via a Catalog Integration that authenticates with a service principal (OAuth). A Snowflake analyst then queries the Snowflake-side Iceberg table references, and the data is read directly from UC managed storage with no copy on either side.

<details id="what-were-building-details">
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Expand to see diagram</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="visibility: hidden">&#x25B6;</span>
      <div style="width: 100%">
<div class="mermaid" id="diagram-3-2-snowflake-architecture" style="font-size: 1em;">
flowchart TB
    subgraph DBX["Databricks (Unity Catalog)"]
        direction TB
        DELTA["<b>store_sales_delta_uniform</b><br/><i>Delta Managed Table<br/>(UniForm enabled)</i>"]
        ICE["<b>store_sales_iceberg</b><br/><i>Iceberg Managed Table</i>"]
    end
    REST["<b>Iceberg REST Catalog</b><br/><i>External Data Access enabled</i>"]
    CINT["<b>unity_catalog_int_oauth</b><br/>Snowflake Catalog Integration<br/><i>(SP / OAuth, vended creds)</i>"]
    subgraph SF["Snowflake"]
        direction TB
        SFICE["<b>store_sales_iceberg</b><br/><i>Iceberg Table reference</i>"]
        SFUNI["<b>store_sales_delta_uniform</b><br/><i>Iceberg Table reference</i>"]
    end
    ANALYST["<b>Snowflake Analyst</b>"]
    DBX  --> REST
    REST -- "SP / OAuth" --> CINT
    CINT --> SF
    ANALYST --> SF
    style DBX fill:#fff5f3,stroke:#FF3621,stroke-width:2px
    style DELTA fill:#ffffff,stroke:#FF3621
    style ICE fill:#ffffff,stroke:#FF3621
    style REST fill:#FF3621,stroke:#CC2B1A,stroke-width:2px,color:#fff
    style CINT fill:#29B5E8,stroke:#0d6e92,stroke-width:2px,color:#fff
    style SF fill:#f3fbff,stroke:#29B5E8,stroke-width:2px
    style SFICE fill:#ffffff,stroke:#29B5E8
    style SFUNI fill:#ffffff,stroke:#29B5E8
    style ANALYST fill:#eceff1,stroke:#37474f,stroke-width:2px
</div>
      </div>
    </div>
  </div>
</details>
<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: false });
const id = "#diagram-3-2-snowflake-architecture";
async function renderDiagram() {
  await mermaid.run({ querySelector: id });
  document.querySelectorAll(id + ' svg text, ' + id + ' svg .nodeLabel, ' + id + ' svg foreignObject div, ' + id + ' svg span').forEach(el => { el.style.fontSize = '1em'; });
}
await new Promise(r => requestAnimationFrame(r));
try { await renderDiagram(); } catch(e) { await new Promise(r => setTimeout(r, 1000)); await renderDiagram(); }
const det = document.getElementById('what-were-building-details');
if (det) {
  det.addEventListener('toggle', async () => {
    if (!det.open) return;
    const node = document.querySelector(id);
    if (node && !node.querySelector('svg')) {
      node.removeAttribute('data-processed');
      await renderDiagram();
    }
  });
}
</script>

In [0]:
%run ../Includes/Classroom-Setup-3-demo

## A. Databricks Preparation

Steps in this section happen on the Databricks side. They expose the UC Iceberg tables to an external engine - in this case, Snowflake - through a service principal that Snowflake will authenticate as.

### A1. (Databricks) Verify the UC Iceberg Tables

The `store_sales_iceberg` and `store_sales_delta_uniform` tables created in Module 2 (or by `Classroom-Setup-3` if Module 2 was skipped) are what Snowflake will consume. Confirm they exist and inspect their format.

In [0]:
SELECT
  table_name,
  data_source_format,
  table_type
FROM system.information_schema.tables
WHERE table_catalog = current_catalog()
  AND table_schema  = current_schema()
  AND table_name IN ('store_sales_iceberg', 'store_sales_delta_uniform')
ORDER BY table_name;

In [0]:
SELECT 'store_sales_iceberg'       AS table_name, COUNT(*) AS row_count FROM store_sales_iceberg
UNION ALL
SELECT 'store_sales_delta_uniform' AS table_name, COUNT(*) AS row_count FROM store_sales_delta_uniform;

### A2. (Databricks) Enable External Data Access on the Metastore

External Iceberg engines reach UC through the Iceberg REST endpoint, which is gated behind a metastore-level switch. Open the pulldown below to see the steps; the screenshot pulldown shows where the toggle lives in Catalog Explorer.

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Metastore Admin Required</strong>
            <p style="margin: 8px 0 0 0; color: #333;">This toggle is only visible to Metastore Admins. If you do not see it, your account does not have the role.</p>
        </div>
    </div>
</div>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Enable External Data Access (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">In the Databricks workspace UI:</p>
    <ol style="margin: 0 0 0 20px; color: #333;">
      <li>Open <strong>Catalog Explorer</strong></li>
      <li>Click the &#9881; icon (top right of the catalog tree) and choose <strong>Metastore</strong></li>
      <li>In the metastore details page, toggle <strong>External data access</strong> to <strong>enabled</strong></li>
    </ol>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Show Screenshot (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/enable-external-access.png" alt="Catalog Explorer metastore settings showing the External data access toggle" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

### A3. (Databricks) Create a Service Principal

A service principal is a headless identity that Snowflake will authenticate as when calling the UC Iceberg REST endpoint. Snowflake never holds a Databricks user token; it holds an SP credential.

<div style="font-size: 1em; border-left: 4px solid #607d8b; background: #eceff1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #37474f; font-size: 1.1em;">Why a service principal?</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Snowflake authenticates non-interactively. A service principal gives you a credential that does not expire on user offboarding, can be revoked independently, and produces audit log entries clearly attributable to the integration rather than to a person.</p>
        </div>
    </div>
</div>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Create the Service Principal (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">In the Databricks workspace UI:</p>
    <ol style="margin: 0 0 0 20px; color: #333;">
      <li>Click the user avatar (top right of the workspace) -&gt; <strong>Settings</strong></li>
      <li>Open <strong>Identity and Access</strong> -&gt; <strong>Service principals</strong> -&gt; <strong>Manage</strong></li>
      <li>Click <strong>Add service principal</strong> -&gt; <strong>Add new</strong></li>
      <li>Enter a display name (for example, <code>snowflake-uc-reader</code>) and create</li>
      <li>From the service principal detail page, copy the <strong>Application ID</strong> - you will paste it into the <code>GRANT</code> statements and into the Snowflake catalog integration</li>
    </ol>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Show Screenshot (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/databricks-service-principal.png" alt="Workspace Settings - Identity and Access - Service principals page" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

### A4. (Databricks) Add the Service Principal to the Workspace

A service principal created in the **account console** exists at the account level - but this does **not** automatically make it a member of any workspace. Until the SP is added as a workspace user, the workspace's OIDC token endpoint (`/oidc/v1/token`) will reject its credentials with `invalid_client: Client authentication failed`, even if the App ID and secret are correct.

This is purely an account -&gt; workspace federation step. No new identity is created; the existing account SP just gets enrolled in this workspace. After this, the OAuth client credentials flow against this workspace will start succeeding immediately.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Add SP to Workspace (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">In the Account UI:</p>
    <ol style="margin: 0 0 0 20px; color: #333;">
      <li>Select <strong>Workspaces</strong></li>
      <li>Open your target workspace</li>
      <li>Go to the <strong>Permissions</strong> tab</li>
      <li>Click <strong>Add permissions</strong> and search for the service principal by name or App ID</li>
      <li>Select it and click <strong>Add</strong></li>
    </ol>
    <p style="margin: 12px 0 0 0; color: #333;">The SP should now appear in the workspace's Users list.</p>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Show Screenshot (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/add-service-principal-to-workspace.png" alt="Workspace Settings - Identity and Access - Users - Add existing dialog showing the account service principal being added" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

### A5. (Databricks) Generate an OAuth Secret for the Service Principal

The catalog integration in Snowflake uses an OAuth client credentials flow. The client ID is the SP's Application ID (from A3); the client secret is generated here.

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #c62828; font-size: 1.1em;">Treat This Secret Like a Password</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Never paste it into a notebook cell, a chat, or a ticket. Store it in a password manager or secrets vault, and rotate it on a schedule. If it leaks, revoke it from the service principal's Secrets tab and generate a new one.</p>
        </div>
    </div>
</div>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Generate OAuth Secret (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">In the Databricks workspace UI:</p>
    <ol style="margin: 0 0 0 20px; color: #333;">
      <li>On the service principal detail page, open the <strong>Secrets</strong> tab</li>
      <li>Click <strong>Generate secret</strong>, choose a <strong>lifetime</strong> (rotate before expiry), then <strong>Generate</strong></li>
      <li><strong>Copy the secret immediately</strong> - it is shown exactly once</li>
    </ol>
  </div>
</details>

### A6. (Databricks) Grant Privileges to the Service Principal

The SP now exists but holds no UC privileges. Snowflake's catalog integration will not work until the SP can reach the catalog, the schema, the tables, and - critically - is allowed to use the schema *externally* (the <span style="color: #c2410c; font-weight: bold; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; background: #f5f5f5; padding: 2px 6px; border-radius: 3px">EXTERNAL USE SCHEMA</span> privilege is the gate that distinguishes external Iceberg REST clients from internal Databricks compute).

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Before Running</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Replace the value for <code>sp_uuid</code> in the cell below with the Application ID copied from A3. The catalog and schema names are resolved automatically from the current session by <code>current_catalog()</code> / <code>current_schema()</code>.</p>
        </div>
    </div>
</div>

In [0]:
-- Run as Metastore Admin / Account Admin.
-- Substitute the SP application ID below with the value from step A3.
--
BEGIN
  DECLARE sp_uuid STRING DEFAULT '09532644-0b2e-4576-804e-43839bdcadd5'; -- replace with your SP app ID
  DECLARE cat STRING DEFAULT 'instructor_interop_demo';
  DECLARE sch STRING DEFAULT 'data_interoperability_tpcds';

  -- 1. Allow the SP to read the schema through external Iceberg REST.
  EXECUTE IMMEDIATE
    'GRANT EXTERNAL USE SCHEMA ON CATALOG `' || cat || '` TO `' || sp_uuid || '`';

  -- 2. Standard catalog/schema traversal.
  EXECUTE IMMEDIATE
    'GRANT USE CATALOG ON CATALOG `' || cat || '` TO `' || sp_uuid || '`';
  EXECUTE IMMEDIATE
    'GRANT USE SCHEMA ON SCHEMA `' || cat || '`.`' || sch || '` TO `' || sp_uuid || '`';

  -- 3. SELECT on the specific tables Snowflake will read.
  EXECUTE IMMEDIATE
    'GRANT SELECT ON TABLE `' || cat || '`.`' || sch || '`.store_sales_iceberg TO `' || sp_uuid || '`';
  EXECUTE IMMEDIATE
    'GRANT SELECT ON TABLE `' || cat || '`.`' || sch || '`.store_sales_delta_uniform TO `' || sp_uuid || '`';
END;

In [0]:
-- SHOW GRANTS on schema
-- Substitute the SP application ID below to match the value used above.
SHOW GRANTS `09532644-0b2e-4576-804e-43839bdcadd5` -- replace with your SP app ID
  ON SCHEMA instructor_interop_demo.data_interoperability_tpcds;

In [0]:
-- SHOW GRANTS on tables
-- Substitute the SP application ID below to match the value used above.
SELECT
  grantee as Principal,
  privilege_type as ActionType,
  'TABLE' as ObjectType,
  table_catalog || '.' || table_schema || '.' || table_name as ObjectKey
FROM instructor_interop_demo.information_schema.table_privileges
WHERE table_catalog = 'instructor_interop_demo'
  AND table_schema  = 'data_interoperability_tpcds'
  AND table_name IN ('store_sales_iceberg', 'store_sales_delta_uniform')
  AND grantee = '09532644-0b2e-4576-804e-43839bdcadd5' -- replace with your SP app ID
ORDER BY table_name, privilege_type;

## B. Snowflake Preparation

The following steps are run from a Snowflake worksheet as <code>ACCOUNTADMIN</code>. They create the catalog integration that points at UC's Iceberg REST endpoint, then create Snowflake-side Iceberg tables that read directly from UC managed storage.

### B1. (Snowflake) Create the Catalog Integration

The catalog integration is Snowflake's named handle to an external Iceberg catalog. It bundles the REST endpoint, the OAuth credentials, and the credential-vending mode.

Substitutions (from the cells above):
- `{workspace_host}` - the workspace URL (e.g. `dbc-12345abc-6789.cloud.databricks.com`)
- `{schema}` - the value of `current_schema()` from the setup cell output (`data_interoperability_tpcds`)
- `{catalog}` - the value of `current_catalog()` from the setup cell output (your `labuser_*` catalog)
- `{service_principal_app_id}` - Application ID from A3
- `{service_principal_secret}` - secret from A5

Note `ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS`: with this set, UC vends short-lived credentials for the underlying object storage to Snowflake at query time. Snowflake never holds a long-lived cloud storage credential, and you do not need to provision a separate `EXTERNAL VOLUME` with its own IAM role.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Create Catalog Integration (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
-- Run as ACCOUNTADMIN
CREATE OR REPLACE CATALOG INTEGRATION unity_catalog_int_oauth
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  CATALOG_NAMESPACE = '{schema}'
  REST_CONFIG = (
    CATALOG_URI = '{workspace_host}/api/2.1/unity-catalog/iceberg-rest',
    CATALOG_NAME = '{catalog}',
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  REST_AUTHENTICATION = (
    TYPE = OAUTH,
    OAUTH_TOKEN_URI     = '{workspace_host}/oidc/v1/token',
    OAUTH_CLIENT_ID     = '{service_principal_app_id}',
    OAUTH_CLIENT_SECRET = '{service_principal_secret}',
    OAUTH_ALLOWED_SCOPES = ('all-apis', 'sql')
  )
  ENABLED = TRUE
  REFRESH_INTERVAL_SECONDS = 30;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Snowflake';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### B2. (Snowflake) Define Iceberg Tables

Each Snowflake `ICEBERG TABLE` is a thin reference to a UC table reached through the catalog integration. There is no copy of the data in Snowflake - only metadata. `AUTO_REFRESH = TRUE` makes Snowflake re-poll the UC catalog at the integration's `REFRESH_INTERVAL_SECONDS` (30s above) so new snapshots written from Databricks appear without an explicit refresh.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Create Iceberg Table References (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
-- Native managed Iceberg
CREATE OR REPLACE ICEBERG TABLE store_sales_iceberg
  CATALOG = 'unity_catalog_int_oauth'
  CATALOG_TABLE_NAME = 'store_sales_iceberg'
  AUTO_REFRESH = TRUE;

-- Delta + UniForm (Iceberg-readable Delta)
CREATE OR REPLACE ICEBERG TABLE store_sales_delta_uniform
  CATALOG = 'unity_catalog_int_oauth'
  CATALOG_TABLE_NAME = 'store_sales_delta_uniform'
  AUTO_REFRESH = TRUE;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Snowflake';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### B3. (Snowflake) Query the UC Tables

Smoke-test that both tables resolve, then run a cross-platform analytic query that joins the UC fact table with Snowflake's resident sample dimensions. The query derives its date range at runtime from `store_sales_iceberg` (a CTE that itself is a cross-platform join), so it returns rows regardless of which days the source sample happens to contain. Neither side copies data; the UC table is read live through the catalog integration.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Smoke Test Queries (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
-- Both should return 1,000,000
SELECT COUNT(*) AS row_count FROM store_sales_iceberg;
SELECT COUNT(*) AS row_count FROM store_sales_delta_uniform;

-- Schema check
DESCRIBE TABLE store_sales_iceberg;
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/snowflake/29B5E8" width="20" height="20" style="vertical-align: middle;"> Snowflake:</span> Cross-Platform Analytic Query (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
-- Cross-platform analytic: top 5 item categories by net revenue across the
-- date range present in store_sales_iceberg, joining UC's fact data with
-- Snowflake's resident TPC-DS sample dimensions.
WITH sample_dates AS (
  SELECT
    MIN(d.d_date) AS min_date,
    MAX(d.d_date) AS max_date
  FROM store_sales_iceberg s
  JOIN SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.DATE_DIM d
    ON d.d_date_sk = s.ss_sold_date_sk
)
SELECT
  i.i_category,
  COUNT(*)                        AS line_items,
  SUM(s.ss_quantity)              AS units_sold,
  ROUND(SUM(s.ss_net_paid), 2)    AS net_revenue,
  ROUND(AVG(s.ss_sales_price), 2) AS avg_unit_price,
  (SELECT min_date || ' to ' || max_date FROM sample_dates) AS date_range
FROM store_sales_iceberg s
JOIN SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.DATE_DIM d ON d.d_date_sk = s.ss_sold_date_sk
JOIN SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.ITEM     i ON i.i_item_sk = s.ss_item_sk
GROUP BY i.i_category
ORDER BY net_revenue DESC
LIMIT 5;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Snowflake';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## Key Takeaways

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">What This Demo Shows</strong>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li>External Iceberg engines reach UC through the <strong>Iceberg REST endpoint</strong>, gated by a metastore-level <strong>External Data Access</strong> toggle</li>
                <li>Authentication uses a Databricks <strong>service principal</strong> with an OAuth secret - never an interactive user token</li>
                <li><strong>EXTERNAL USE SCHEMA</strong> is the load-bearing privilege - without it the SP can read internally but not via Iceberg REST</li>
                <li>Snowflake's <strong>CATALOG INTEGRATION</strong> with <code>VENDED_CREDENTIALS</code> lets UC vend short-lived storage credentials at query time, removing the need for a Snowflake-side <code>EXTERNAL VOLUME</code> + IAM role</li>
                <li>The same pattern works for <strong>Delta + UniForm</strong>: external Iceberg engines see Iceberg metadata regardless of the underlying format</li>
                <li>UC remains the single governance authority - permissions, lineage, and audit are unchanged regardless of which engine queries the data</li>
            </ul>
        </div>
    </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>